# Module 5.1: 迁移学习与微调基础

## 1. 本章概览

### 📚 学习目标

1. **迁移学习理论**：理解为什么预训练模型有效
2. **预训练-微调范式**：掌握现代NLP的核心方法论
3. **特征提取 vs 微调**：理解两种方法的区别和适用场景
4. **学习率调度**：掌握微调中的关键训练技巧
5. **超参数选择**：学会为微调任务选择合适的超参数
6. **完整流程**：实现端到端的文本分类微调

### 🎯 核心问题

- 为什么预训练模型能迁移到新任务？
- 什么时候用特征提取，什么时候用微调？
- 微调时学习率应该设置多大？
- 如何避免微调时的过拟合？
- 如何评估微调效果？

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 新增业务标签后重训时间太长？→ 迁移学习冻结策略与分层学习率
- 预训练模型直接用效果差怎么办？→ 领域适配与微调的必要性

### 🗺️ 知识地图

```
迁移学习基础
    ↓
预训练-微调范式
├── 特征提取（冻结参数）
└── 微调（更新参数）
    ↓
微调策略
├── 学习率调度
├── 判别式学习率
└── 渐进式解冻
    ↓
超参数优化
    ↓
完整微调流程
```

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import time

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("✓ Libraries imported")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. 迁移学习理论

### 2.1 什么是迁移学习？

**定义**：将从源任务（source task）学到的知识应用到目标任务（target task）的过程。

**核心思想**：
- 不同任务之间存在共享的知识
- 在大规模数据上学到的表示可以迁移
- 避免从零开始训练

### 2.2 为什么迁移学习有效？

**1. 层次化特征学习**

神经网络学习层次化的表示：
- **底层**：通用特征（词嵌入、语法结构）
- **中层**：语义特征（短语、句法关系）
- **高层**：任务特定特征

**2. 数据效率**

预训练模型已经学到了语言的基本规律，微调只需要少量标注数据。

**3. 泛化能力**

在大规模语料上预训练的模型具有更好的泛化能力。

### 2.3 迁移学习的三种场景

| 场景 | 源域数据 | 目标域数据 | 策略 |
|------|---------|-----------|------|
| **场景1** | 大量 | 大量 | 预训练 + 全量微调 |
| **场景2** | 大量 | 少量 | 预训练 + 轻量微调/特征提取 |
| **场景3** | 大量 | 极少 | 预训练 + Few-shot学习 |

### 2.4 数学表示

**预训练阶段**：
$$\theta_{\text{pretrained}} = \arg\min_{\theta} \mathcal{L}_{\text{pretrain}}(\mathcal{D}_{\text{source}}, \theta)$$

**微调阶段**：
$$\theta_{\text{finetuned}} = \arg\min_{\theta} \mathcal{L}_{\text{task}}(\mathcal{D}_{\text{target}}, \theta)$$

其中 $\theta$ 初始化为 $\theta_{\text{pretrained}}$

In [ ]:
# 🔬 Micro Practice: 迁移学习优势演示
# Goal: 对比从零训练 vs 迁移学习的性能
# Expected outcome: 理解迁移学习的数据效率优势

class SimpleClassifier(nn.Module):
    """简单的分类器用于演示"""
    
    def __init__(self, input_dim=128, hidden_dim=64, num_classes=3):
        super(SimpleClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

def train_model(model, train_loader, epochs=10, lr=0.001):
    """训练模型"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        losses.append(avg_loss)
    
    return losses

def evaluate_model(model, test_loader):
    """评估模型"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    
    return correct / total

# 模拟数据
print("=== 迁移学习 vs 从零训练 ===")
print()

# 源任务：大量数据
X_source = torch.randn(1000, 128)
y_source = torch.randint(0, 3, (1000,))
source_dataset = TensorDataset(X_source, y_source)
source_loader = DataLoader(source_dataset, batch_size=32, shuffle=True)

# 目标任务：少量数据
X_target_train = torch.randn(100, 128)
y_target_train = torch.randint(0, 3, (100,))
target_train_dataset = TensorDataset(X_target_train, y_target_train)
target_train_loader = DataLoader(target_train_dataset, batch_size=16, shuffle=True)

X_target_test = torch.randn(200, 128)
y_target_test = torch.randint(0, 3, (200,))
target_test_dataset = TensorDataset(X_target_test, y_target_test)
target_test_loader = DataLoader(target_test_dataset, batch_size=32)

print(f"源任务数据: {len(X_source)} 样本")
print(f"目标任务训练数据: {len(X_target_train)} 样本")
print(f"目标任务测试数据: {len(X_target_test)} 样本")
print()

# 方法1: 从零训练（只用目标任务数据）
print("方法1: 从零训练")
model_scratch = SimpleClassifier()
losses_scratch = train_model(model_scratch, target_train_loader, epochs=20, lr=0.001)
acc_scratch = evaluate_model(model_scratch, target_test_loader)
print(f"  测试准确率: {acc_scratch:.3f}")
print()

# 方法2: 迁移学习（先在源任务预训练，再在目标任务微调）
print("方法2: 迁移学习")
model_transfer = SimpleClassifier()

# 步骤1: 在源任务上预训练
print("  步骤1: 源任务预训练...")
_ = train_model(model_transfer, source_loader, epochs=10, lr=0.001)

# 步骤2: 在目标任务上微调
print("  步骤2: 目标任务微调...")
losses_transfer = train_model(model_transfer, target_train_loader, epochs=20, lr=0.0001)
acc_transfer = evaluate_model(model_transfer, target_test_loader)
print(f"  测试准确率: {acc_transfer:.3f}")
print()

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练曲线
axes[0].plot(losses_scratch, label='从零训练', linewidth=2, marker='o', markersize=4)
axes[0].plot(losses_transfer, label='迁移学习', linewidth=2, marker='s', markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练曲线对比', fontsize=14, weight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 准确率对比
methods = ['从零训练', '迁移学习']
accuracies = [acc_scratch, acc_transfer]
colors = ['coral', 'steelblue']

bars = axes[1].bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('测试准确率', fontsize=12)
axes[1].set_title('性能对比', fontsize=14, weight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=12, weight='bold')

plt.tight_layout()
plt.show()

improvement = (acc_transfer - acc_scratch) / acc_scratch * 100
print(f"性能提升: {improvement:+.1f}%")
print()
print("核心观察：")
print("- 迁移学习在少量数据上表现更好")
print("- 预训练提供了更好的初始化")
print("- 训练曲线更稳定，收敛更快")
print()
print("✓ 迁移学习优势演示完成！")

## 3. 预训练-微调范式

### 3.1 两阶段学习

现代NLP的标准范式：

```
阶段1: 预训练 (Pre-training)
├── 数据：大规模无标注文本
├── 任务：自监督学习（MLM, NSP, CLM等）
├── 目标：学习通用语言表示
└── 输出：预训练模型参数 θ_pretrained

阶段2: 微调 (Fine-tuning)
├── 数据：少量任务特定标注数据
├── 任务：监督学习（分类、NER、QA等）
├── 目标：适应特定任务
└── 输出：微调模型参数 θ_finetuned
```

### 3.2 预训练任务

**1. Masked Language Modeling (MLM)**
- 随机遮盖15%的token
- 预测被遮盖的token
- 用于BERT等编码器模型

**2. Causal Language Modeling (CLM)**
- 预测下一个token
- 用于GPT等解码器模型

**3. Sequence-to-Sequence (Seq2Seq)**
- 输入序列生成输出序列
- 用于T5等编码器-解码器模型

### 3.3 微调策略

**策略1: 全量微调 (Full Fine-tuning)**
- 更新所有模型参数
- 适用于数据充足的场景
- 性能最好但成本最高

**策略2: 特征提取 (Feature Extraction)**
- 冻结预训练模型
- 只训练任务特定层
- 适用于数据稀缺或计算受限

**策略3: 部分微调 (Partial Fine-tuning)**
- 冻结底层，微调高层
- 平衡性能和效率
- 适用于中等数据量

In [ ]:
# 🔬 Micro Practice: 预训练-微调流程演示
# Goal: 完整演示预训练和微调的两阶段过程
# Expected outcome: 理解预训练如何帮助下游任务

class PretrainedModel(nn.Module):
    """模拟预训练模型"""
    
    def __init__(self, input_dim=100, hidden_dim=64):
        super(PretrainedModel, self).__init__()
        # Encoder (预训练部分)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
    
    def forward(self, x):
        return self.encoder(x)

class FinetunedModel(nn.Module):
    """在预训练模型基础上添加任务头"""
    
    def __init__(self, pretrained_model, num_classes=2):
        super(FinetunedModel, self).__init__()
        self.encoder = pretrained_model.encoder
        # 任务特定的分类头
        self.classifier = nn.Linear(64, num_classes)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

print("=== 预训练-微调两阶段流程 ===")
print()

# 阶段1: 预训练（模拟自监督学习）
print("阶段1: 预训练")
print("-" * 50)

# 创建预训练模型
pretrained_model = PretrainedModel(input_dim=100, hidden_dim=64)

# 模拟预训练数据（大量无标注数据）
X_pretrain = torch.randn(5000, 100)
# 自监督任务：重构输入（简化示例）
pretrain_dataset = TensorDataset(X_pretrain, X_pretrain)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=64, shuffle=True)

# 预训练
criterion_pretrain = nn.MSELoss()
optimizer_pretrain = optim.Adam(pretrained_model.parameters(), lr=0.001)

print(f"预训练数据: {len(X_pretrain)} 样本（无标注）")
print("预训练任务: 自监督学习（重构）")
print("训练中...")

pretrain_losses = []
for epoch in range(5):
    total_loss = 0
    for X_batch, _ in pretrain_loader:
        optimizer_pretrain.zero_grad()
        # 简化的自监督任务
        features = pretrained_model(X_batch)
        # 添加一个重构层用于预训练
        reconstructed = nn.Linear(64, 100).to(features.device)(features)
        loss = criterion_pretrain(reconstructed, X_batch)
        loss.backward()
        optimizer_pretrain.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(pretrain_loader)
    pretrain_losses.append(avg_loss)
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/5, Loss: {avg_loss:.4f}")

print("✓ 预训练完成")
print()

# 阶段2: 微调（监督学习）
print("阶段2: 微调")
print("-" * 50)

# 创建微调模型（使用预训练的encoder）
finetuned_model = FinetunedModel(pretrained_model, num_classes=2)

# 微调数据（少量标注数据）
X_finetune = torch.randn(200, 100)
y_finetune = torch.randint(0, 2, (200,))
finetune_dataset = TensorDataset(X_finetune, y_finetune)
finetune_loader = DataLoader(finetune_dataset, batch_size=16, shuffle=True)

# 测试数据
X_test = torch.randn(100, 100)
y_test = torch.randint(0, 2, (100,))
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"微调数据: {len(X_finetune)} 样本（有标注）")
print("微调任务: 二分类")
print("训练中...")

# 微调（使用较小的学习率）
criterion_finetune = nn.CrossEntropyLoss()
optimizer_finetune = optim.Adam(finetuned_model.parameters(), lr=0.0001)

finetune_losses = []
finetune_accs = []

for epoch in range(10):
    # Training
    finetuned_model.train()
    total_loss = 0
    for X_batch, y_batch in finetune_loader:
        optimizer_finetune.zero_grad()
        outputs = finetuned_model(X_batch)
        loss = criterion_finetune(outputs, y_batch)
        loss.backward()
        optimizer_finetune.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(finetune_loader)
    finetune_losses.append(avg_loss)
    
    # Evaluation
    finetuned_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = finetuned_model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    
    acc = correct / total
    finetune_accs.append(acc)
    
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1}/10, Loss: {avg_loss:.4f}, Acc: {acc:.3f}")

print("✓ 微调完成")
print()

# 可视化两阶段流程
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 预训练损失
axes[0].plot(range(1, 6), pretrain_losses, 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Reconstruction Loss', fontsize=12)
axes[0].set_title('阶段1: 预训练', fontsize=14, weight='bold')
axes[0].grid(True, alpha=0.3)

# 微调损失
axes[1].plot(range(1, 11), finetune_losses, 's-', linewidth=2, markersize=6, color='coral')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Classification Loss', fontsize=12)
axes[1].set_title('阶段2: 微调（损失）', fontsize=14, weight='bold')
axes[1].grid(True, alpha=0.3)

# 微调准确率
axes[2].plot(range(1, 11), finetune_accs, '^-', linewidth=2, markersize=6, color='green')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Accuracy', fontsize=12)
axes[2].set_title('阶段2: 微调（准确率）', fontsize=14, weight='bold')
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("核心观察：")
print("- 预训练在大量无标注数据上学习通用表示")
print("- 微调在少量标注数据上适应特定任务")
print("- 两阶段结合实现高效的知识迁移")
print()
print("✓ 预训练-微调流程演示完成！")

## 4. 特征提取 vs 微调对比

### 4.1 特征提取 (Feature Extraction)

**定义**：冻结预训练模型的所有参数，只训练新添加的任务特定层。

**优点**：
- ✅ 训练速度快（参数少）
- ✅ 内存占用小
- ✅ 不易过拟合
- ✅ 保留预训练知识

**缺点**：
- ❌ 性能可能不如微调
- ❌ 无法适应领域差异大的任务

**适用场景**：
- 数据量很少（<1000样本）
- 计算资源受限
- 任务与预训练任务相似

### 4.2 微调 (Fine-tuning)

**定义**：更新预训练模型的部分或全部参数。

**优点**：
- ✅ 性能通常更好
- ✅ 能适应领域差异
- ✅ 灵活性高

**缺点**：
- ❌ 训练时间长
- ❌ 内存占用大
- ❌ 可能过拟合
- ❌ 可能遗忘预训练知识

**适用场景**：
- 数据量充足（>10000样本）
- 追求最佳性能
- 任务与预训练任务差异大

### 4.3 对比表格

| 维度 | 特征提取 | 微调 |
|------|---------|------|
| **可训练参数** | 仅任务头（~1%） | 全部或部分（10-100%） |
| **训练时间** | 快 | 慢 |
| **内存需求** | 低 | 高 |
| **性能** | 中等 | 最佳 |
| **过拟合风险** | 低 | 中-高 |
| **数据需求** | 少 | 多 |
| **领域适应** | 弱 | 强 |

In [ ]:
# 🔬 Micro Practice: 特征提取 vs 微调性能对比
# Goal: 对比两种方法在不同数据量下的性能
# Expected outcome: 理解何时使用特征提取，何时使用微调

class TransferModel(nn.Module):
    """可配置的迁移学习模型"""
    
    def __init__(self, input_dim=50, hidden_dim=32, num_classes=2):
        super(TransferModel, self).__init__()
        # 预训练的特征提取器
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # 任务特定的分类头
        self.classifier = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)
    
    def freeze_features(self):
        """冻结特征提取器（特征提取模式）"""
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
    
    def unfreeze_features(self):
        """解冻特征提取器（微调模式）"""
        for param in self.feature_extractor.parameters():
            param.requires_grad = True

def train_and_evaluate(model, train_loader, test_loader, epochs=20, lr=0.001):
    """训练并评估模型"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    train_losses = []
    test_accs = []
    train_times = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        start_time = time.time()
        total_loss = 0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        epoch_time = time.time() - start_time
        train_times.append(epoch_time)
        train_losses.append(total_loss / len(train_loader))
        
        # Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
        
        test_accs.append(correct / total)
    
    return train_losses, test_accs, train_times

print("=== 特征提取 vs 微调对比实验 ===")
print()

# 测试不同数据量
data_sizes = [50, 100, 500, 1000]
results = {'feature_extraction': {}, 'fine_tuning': {}}

# 固定测试集
X_test = torch.randn(200, 50)
y_test = torch.randint(0, 2, (200,))
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32)

for size in data_sizes:
    print(f"数据量: {size} 样本")
    print("-" * 50)
    
    # 生成训练数据
    X_train = torch.randn(size, 50)
    y_train = torch.randint(0, 2, (size,))
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=min(16, size), shuffle=True)
    
    # 方法1: 特征提取
    print("  方法1: 特征提取（冻结特征提取器）")
    model_fe = TransferModel()
    # 模拟预训练
    with torch.no_grad():
        for param in model_fe.feature_extractor.parameters():
            param.data = torch.randn_like(param.data) * 0.1
    
    model_fe.freeze_features()
    trainable_fe = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
    print(f"    可训练参数: {trainable_fe}")
    
    losses_fe, accs_fe, times_fe = train_and_evaluate(model_fe, train_loader, test_loader, epochs=20, lr=0.001)
    print(f"    最终准确率: {accs_fe[-1]:.3f}")
    print(f"    平均训练时间: {np.mean(times_fe):.4f}s/epoch")
    
    results['feature_extraction'][size] = {
        'acc': accs_fe[-1],
        'time': np.mean(times_fe),
        'params': trainable_fe
    }
    
    # 方法2: 微调
    print("  方法2: 微调（更新所有参数）")
    model_ft = TransferModel()
    # 使用相同的预训练初始化
    model_ft.load_state_dict(model_fe.state_dict())
    model_ft.unfreeze_features()
    trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
    print(f"    可训练参数: {trainable_ft}")
    
    losses_ft, accs_ft, times_ft = train_and_evaluate(model_ft, train_loader, test_loader, epochs=20, lr=0.0001)
    print(f"    最终准确率: {accs_ft[-1]:.3f}")
    print(f"    平均训练时间: {np.mean(times_ft):.4f}s/epoch")
    
    results['fine_tuning'][size] = {
        'acc': accs_ft[-1],
        'time': np.mean(times_ft),
        'params': trainable_ft
    }
    
    print()

# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 准确率 vs 数据量
fe_accs = [results['feature_extraction'][s]['acc'] for s in data_sizes]
ft_accs = [results['fine_tuning'][s]['acc'] for s in data_sizes]

axes[0, 0].plot(data_sizes, fe_accs, 'o-', linewidth=2, markersize=8, label='特征提取', color='steelblue')
axes[0, 0].plot(data_sizes, ft_accs, 's-', linewidth=2, markersize=8, label='微调', color='coral')
axes[0, 0].set_xlabel('训练样本数', fontsize=11)
axes[0, 0].set_ylabel('测试准确率', fontsize=11)
axes[0, 0].set_title('准确率 vs 数据量', fontsize=13, weight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. 训练时间对比
fe_times = [results['feature_extraction'][s]['time'] for s in data_sizes]
ft_times = [results['fine_tuning'][s]['time'] for s in data_sizes]

x = np.arange(len(data_sizes))
width = 0.35

axes[0, 1].bar(x - width/2, fe_times, width, label='特征提取', color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].bar(x + width/2, ft_times, width, label='微调', color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('训练样本数', fontsize=11)
axes[0, 1].set_ylabel('训练时间 (s/epoch)', fontsize=11)
axes[0, 1].set_title('训练时间对比', fontsize=13, weight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(data_sizes)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. 参数量对比
params_fe = results['feature_extraction'][data_sizes[0]]['params']
params_ft = results['fine_tuning'][data_sizes[0]]['params']

methods = ['特征提取', '微调']
params = [params_fe, params_ft]
colors_params = ['steelblue', 'coral']

bars = axes[1, 0].bar(methods, params, color=colors_params, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('可训练参数数量', fontsize=11)
axes[1, 0].set_title('参数量对比', fontsize=13, weight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

for bar, param in zip(bars, params):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{param}', ha='center', va='bottom', fontsize=10, weight='bold')

# 4. 性能提升 vs 数据量
improvements = [(ft - fe) / fe * 100 for fe, ft in zip(fe_accs, ft_accs)]

colors_imp = ['red' if imp < 0 else 'green' for imp in improvements]
axes[1, 1].bar(data_sizes, improvements, color=colors_imp, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1, 1].set_xlabel('训练样本数', fontsize=11)
axes[1, 1].set_ylabel('微调相对提升 (%)', fontsize=11)
axes[1, 1].set_title('微调性能提升', fontsize=13, weight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- 数据少时：特征提取表现更稳定，不易过拟合")
print("- 数据多时：微调性能更好，能充分适应任务")
print("- 训练速度：特征提取更快（参数少）")
print("- 内存占用：特征提取更小（无需存储梯度）")
print()
print("✓ 特征提取 vs 微调对比完成！")

## 5. 学习率调度策略

### 5.1 为什么微调需要特殊的学习率？

**关键原则**：预训练模型的参数已经接近最优，需要小心更新。

**灾难性遗忘 (Catastrophic Forgetting)** 是指模型在学习新任务时，之前学到的知识被严重覆盖或丢失的现象。在本章场景中，它是微调预训练模型时需要重点防范的核心风险，直接影响学习率和解冻策略的选择。

**问题**：
- 学习率太大 → 破坏预训练知识（灾难性遗忘），性能下降
- 学习率太小 → 收敛慢，无法充分适应新任务

**解决方案**：使用动态学习率调度策略

### 5.2 常用学习率调度策略

#### 1. Warmup（预热）

**思想**：从很小的学习率开始，逐渐增加到目标学习率。

**公式**：
$$\text{lr}(t) = \text{lr}_{\text{base}} \times \min\left(1, \frac{t}{T_{\text{warmup}}}\right)$$

**优点**：
- 避免训练初期的不稳定
- 保护预训练权重

#### 2. Linear Decay（线性衰减）

**公式**：
$$\text{lr}(t) = \text{lr}_{\text{base}} \times \left(1 - \frac{t}{T_{\text{total}}}\right)$$

#### 3. Cosine Annealing（余弦退火）

**公式**：
$$\text{lr}(t) = \text{lr}_{\text{min}} + \frac{1}{2}(\text{lr}_{\text{max}} - \text{lr}_{\text{min}})\left(1 + \cos\left(\frac{t\pi}{T_{\text{total}}}\right)\right)$$

**优点**：
- 平滑的学习率变化
- 后期精细调整

#### 4. Discriminative Learning Rates（判别式学习率）

**判别式学习率 (Discriminative Learning Rate)** 是一种为模型不同层设置不同学习率的微调技巧，通常底层使用较小的学习率以保留通用特征，高层使用较大的学习率以快速适应新任务。在本章场景中，它用于在迁移学习微调阶段平衡"保留预训练知识"与"适应新任务"之间的矛盾。

**策略**：
- 底层（接近输入）：小学习率（保留通用特征）
- 高层（接近输出）：大学习率（快速适应任务）

**公式**：
$$\text{lr}_{\text{layer}_i} = \text{lr}_{\text{base}} \times \xi^{L-i}$$

其中 $\xi < 1$ 是衰减因子，$L$ 是总层数，$i$ 是当前层索引。

#### 5. Gradual Unfreezing（渐进式解冻）

**思想**：逐层解冻模型参数。

**步骤**：
1. 只训练顶层（任务头）
2. 解冻最后几层，继续训练
3. 逐步解冻更多层
4. 最终微调整个模型

### 5.3 学习率选择指南

| 场景 | 推荐学习率 | 调度策略 |
|------|-----------|----------|
| 特征提取 | 1e-3 ~ 1e-2 | Constant |
| 微调（数据充足） | 1e-5 ~ 5e-5 | Warmup + Linear Decay |
| 微调（数据稀缺） | 1e-6 ~ 1e-5 | Warmup + Cosine |
| 大模型微调 | 1e-6 ~ 1e-5 | Warmup + Cosine + Discriminative |

In [ ]:
# 🔬 Micro Practice: 学习率调度器实现与可视化
# Goal: 实现并可视化不同的学习率调度策略
# Expected outcome: 理解各种调度策略的特点

class LearningRateScheduler:
    """学习率调度器集合"""
    
    @staticmethod
    def constant(lr_base, step, total_steps):
        """常数学习率"""
        return lr_base
    
    @staticmethod
    def linear_warmup(lr_base, step, warmup_steps):
        """线性预热"""
        if step < warmup_steps:
            return lr_base * (step / warmup_steps)
        return lr_base
    
    @staticmethod
    def linear_decay(lr_base, step, total_steps):
        """线性衰减"""
        return lr_base * (1 - step / total_steps)
    
    @staticmethod
    def warmup_linear_decay(lr_base, step, warmup_steps, total_steps):
        """预热 + 线性衰减"""
        if step < warmup_steps:
            return lr_base * (step / warmup_steps)
        else:
            return lr_base * (1 - (step - warmup_steps) / (total_steps - warmup_steps))
    
    @staticmethod
    def cosine_annealing(lr_base, step, total_steps, lr_min=0):
        """余弦退火"""
        return lr_min + 0.5 * (lr_base - lr_min) * (1 + np.cos(np.pi * step / total_steps))
    
    @staticmethod
    def warmup_cosine(lr_base, step, warmup_steps, total_steps, lr_min=0):
        """预热 + 余弦退火"""
        if step < warmup_steps:
            return lr_base * (step / warmup_steps)
        else:
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return lr_min + 0.5 * (lr_base - lr_min) * (1 + np.cos(np.pi * progress))
    
    @staticmethod
    def exponential_decay(lr_base, step, decay_rate=0.96, decay_steps=100):
        """指数衰减"""
        return lr_base * (decay_rate ** (step / decay_steps))

print("=== 学习率调度策略可视化 ===")
print()

# 参数设置
lr_base = 1e-4
total_steps = 1000
warmup_steps = 100

# 生成学习率曲线
steps = np.arange(total_steps)
schedulers = {
    'Constant': [LearningRateScheduler.constant(lr_base, s, total_steps) for s in steps],
    'Linear Warmup': [LearningRateScheduler.linear_warmup(lr_base, s, warmup_steps) for s in steps],
    'Linear Decay': [LearningRateScheduler.linear_decay(lr_base, s, total_steps) for s in steps],
    'Warmup + Linear': [LearningRateScheduler.warmup_linear_decay(lr_base, s, warmup_steps, total_steps) for s in steps],
    'Cosine Annealing': [LearningRateScheduler.cosine_annealing(lr_base, s, total_steps) for s in steps],
    'Warmup + Cosine': [LearningRateScheduler.warmup_cosine(lr_base, s, warmup_steps, total_steps) for s in steps],
    'Exponential Decay': [LearningRateScheduler.exponential_decay(lr_base, s) for s in steps],
}

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. 所有策略对比
colors = ['gray', 'blue', 'red', 'green', 'orange', 'purple', 'brown']
for (name, lrs), color in zip(schedulers.items(), colors):
    axes[0, 0].plot(steps, lrs, label=name, linewidth=2, color=color, alpha=0.8)

axes[0, 0].set_xlabel('Training Step', fontsize=12)
axes[0, 0].set_ylabel('Learning Rate', fontsize=12)
axes[0, 0].set_title('所有学习率调度策略', fontsize=14, weight='bold')
axes[0, 0].legend(fontsize=9, loc='upper right')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim(0, total_steps)

# 2. Warmup策略对比
warmup_schedulers = {
    'No Warmup': schedulers['Linear Decay'],
    'Warmup + Linear': schedulers['Warmup + Linear'],
    'Warmup + Cosine': schedulers['Warmup + Cosine'],
}

for name, lrs in warmup_schedulers.items():
    axes[0, 1].plot(steps, lrs, label=name, linewidth=2.5, alpha=0.8)

axes[0, 1].axvline(x=warmup_steps, color='red', linestyle='--', linewidth=1.5, label='Warmup End')
axes[0, 1].set_xlabel('Training Step', fontsize=12)
axes[0, 1].set_ylabel('Learning Rate', fontsize=12)
axes[0, 1].set_title('Warmup策略对比', fontsize=14, weight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim(0, 300)  # 放大前期

# 3. 判别式学习率（不同层）
num_layers = 5
decay_factor = 0.8

for layer in range(num_layers):
    layer_lr_base = lr_base * (decay_factor ** (num_layers - 1 - layer))
    layer_lrs = [LearningRateScheduler.warmup_cosine(layer_lr_base, s, warmup_steps, total_steps) for s in steps]
    axes[1, 0].plot(steps, layer_lrs, label=f'Layer {layer+1}', linewidth=2, alpha=0.8)

axes[1, 0].set_xlabel('Training Step', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('判别式学习率（不同层）', fontsize=14, weight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# 4. 渐进式解冻示意图
unfreeze_schedule = [
    (0, 200, 'Layer 5 (Head)'),
    (200, 400, 'Layer 4-5'),
    (400, 600, 'Layer 3-5'),
    (600, 800, 'Layer 2-5'),
    (800, 1000, 'All Layers'),
]

colors_unfreeze = ['red', 'orange', 'yellow', 'lightgreen', 'green']
for (start, end, label), color in zip(unfreeze_schedule, colors_unfreeze):
    axes[1, 1].axvspan(start, end, alpha=0.3, color=color, label=label)
    axes[1, 1].text((start + end) / 2, 0.5, label, ha='center', va='center', 
                   fontsize=9, weight='bold', rotation=0)

axes[1, 1].set_xlabel('Training Step', fontsize=12)
axes[1, 1].set_ylabel('Active Layers', fontsize=12)
axes[1, 1].set_title('渐进式解冻策略', fontsize=14, weight='bold')
axes[1, 1].set_xlim(0, total_steps)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_yticks([])
axes[1, 1].legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

print("核心观察：")
print("- Warmup：避免训练初期的不稳定，保护预训练权重")
print("- Cosine：平滑衰减，后期精细调整")
print("- 判别式LR：底层小LR保留通用特征，高层大LR快速适应")
print("- 渐进解冻：逐层适应，避免灾难性遗忘")
print()
print("✓ 学习率调度策略演示完成！")

## 6. 超参数选择指南

### 6.1 关键超参数

微调中需要调整的主要超参数：

#### 1. 学习率 (Learning Rate)

**最重要的超参数！**

**推荐范围**：
- 特征提取：`1e-3` ~ `1e-2`
- 微调（BERT-base）：`2e-5` ~ `5e-5`
- 微调（大模型）：`1e-6` ~ `1e-5`

**调优策略**：
- 从小开始，逐步增大
- 观察训练曲线是否稳定
- 使用学习率查找器（LR Finder）

#### 2. 批次大小 (Batch Size)

**推荐范围**：`8` ~ `32`

**权衡**：
- 大批次：训练稳定，但可能泛化差
- 小批次：泛化好，但训练不稳定

**技巧**：
- GPU内存不足时使用梯度累积
- 批次大小加倍时，学习率也可以适当增大

#### 3. 训练轮数 (Epochs)

**推荐范围**：`3` ~ `10`

**注意**：
- 微调通常不需要太多轮
- 使用早停（Early Stopping）防止过拟合
- 监控验证集性能

#### 4. 权重衰减 (Weight Decay)

**推荐范围**：`0.01` ~ `0.1`

**作用**：L2正则化，防止过拟合

**注意**：
- 不要对偏置和LayerNorm参数应用
- 数据少时增大权重衰减

#### 5. Dropout

**推荐范围**：`0.1` ~ `0.3`

**策略**：
- 数据充足：使用预训练模型的默认值
- 数据稀缺：增大dropout（0.3-0.5）

#### 6. Warmup Steps

**推荐范围**：总步数的 `5%` ~ `10%`

**计算**：
```python
total_steps = (len(train_dataset) // batch_size) * epochs
warmup_steps = int(0.1 * total_steps)
```

### 6.2 超参数调优策略

#### 策略1: 网格搜索 (Grid Search)

**优点**：全面
**缺点**：计算成本高

#### 策略2: 随机搜索 (Random Search)

**优点**：效率高
**缺点**：可能错过最优组合

#### 策略3: 贝叶斯优化 (Bayesian Optimization)

**优点**：智能搜索
**缺点**：实现复杂

#### 策略4: 经验法则（推荐）

**步骤**：
1. 使用推荐的默认值
2. 只调整学习率（最重要）
3. 如果过拟合，增大正则化
4. 如果欠拟合，增大模型容量或训练轮数

### 6.3 超参数配置示例

#### 配置1: 数据充足（>10K样本）

```python
config = {
    'learning_rate': 3e-5,
    'batch_size': 32,
    'epochs': 3,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'dropout': 0.1,
}
```

#### 配置2: 数据稀缺（<1K样本）

```python
config = {
    'learning_rate': 1e-5,
    'batch_size': 8,
    'epochs': 10,
    'weight_decay': 0.1,
    'warmup_ratio': 0.1,
    'dropout': 0.3,
}
```

#### 配置3: 大模型（>1B参数）

```python
config = {
    'learning_rate': 1e-6,
    'batch_size': 4,  # 使用梯度累积
    'gradient_accumulation_steps': 8,
    'epochs': 3,
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,
}
```

In [ ]:
# 🔬 Micro Practice: 超参数敏感性分析
# Goal: 分析不同超参数对性能的影响
# Expected outcome: 理解哪些超参数最重要

def train_with_config(config, X_train, y_train, X_val, y_val):
    """使用指定配置训练模型"""
    model = SimpleClassifier(input_dim=50, hidden_dim=32, num_classes=2)
    
    # 数据加载
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    
    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=32)
    
    # 优化器
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    
    # 训练
    for epoch in range(config['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
        
        # 验证
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
        
        val_acc = correct / total
        best_val_acc = max(best_val_acc, val_acc)
    
    return best_val_acc

print("=== 超参数敏感性分析 ===")
print()

# 准备数据
X_train = torch.randn(500, 50)
y_train = torch.randint(0, 2, (500,))
X_val = torch.randn(100, 50)
y_val = torch.randint(0, 2, (100,))

# 基准配置
base_config = {
    'learning_rate': 1e-4,
    'batch_size': 16,
    'epochs': 10,
    'weight_decay': 0.01,
}

# 1. 学习率敏感性
print("1. 学习率敏感性分析")
print("-" * 50)
learning_rates = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
lr_results = []

for lr in learning_rates:
    config = base_config.copy()
    config['learning_rate'] = lr
    acc = train_with_config(config, X_train, y_train, X_val, y_val)
    lr_results.append(acc)
    print(f"  LR={lr:.0e}: Val Acc={acc:.3f}")

print()

# 2. 批次大小敏感性
print("2. 批次大小敏感性分析")
print("-" * 50)
batch_sizes = [4, 8, 16, 32, 64]
bs_results = []

for bs in batch_sizes:
    config = base_config.copy()
    config['batch_size'] = bs
    acc = train_with_config(config, X_train, y_train, X_val, y_val)
    bs_results.append(acc)
    print(f"  Batch Size={bs}: Val Acc={acc:.3f}")

print()

# 3. 权重衰减敏感性
print("3. 权重衰减敏感性分析")
print("-" * 50)
weight_decays = [0, 0.001, 0.01, 0.1, 0.5]
wd_results = []

for wd in weight_decays:
    config = base_config.copy()
    config['weight_decay'] = wd
    acc = train_with_config(config, X_train, y_train, X_val, y_val)
    wd_results.append(acc)
    print(f"  Weight Decay={wd}: Val Acc={acc:.3f}")

print()

# 4. 训练轮数敏感性
print("4. 训练轮数敏感性分析")
print("-" * 50)
epoch_nums = [3, 5, 10, 15, 20]
epoch_results = []

for ep in epoch_nums:
    config = base_config.copy()
    config['epochs'] = ep
    acc = train_with_config(config, X_train, y_train, X_val, y_val)
    epoch_results.append(acc)
    print(f"  Epochs={ep}: Val Acc={acc:.3f}")

print()

# 可视化敏感性分析
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 学习率
axes[0, 0].plot([f'{lr:.0e}' for lr in learning_rates], lr_results, 'o-', 
               linewidth=2, markersize=10, color='steelblue')
axes[0, 0].set_xlabel('Learning Rate', fontsize=11)
axes[0, 0].set_ylabel('Validation Accuracy', fontsize=11)
axes[0, 0].set_title('学习率敏感性', fontsize=13, weight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 标注最佳值
best_idx = np.argmax(lr_results)
axes[0, 0].scatter([best_idx], [lr_results[best_idx]], s=200, c='red', 
                  marker='*', zorder=5, label='Best')
axes[0, 0].legend()

# 2. 批次大小
axes[0, 1].plot(batch_sizes, bs_results, 's-', linewidth=2, markersize=10, color='coral')
axes[0, 1].set_xlabel('Batch Size', fontsize=11)
axes[0, 1].set_ylabel('Validation Accuracy', fontsize=11)
axes[0, 1].set_title('批次大小敏感性', fontsize=13, weight='bold')
axes[0, 1].grid(True, alpha=0.3)

best_idx = np.argmax(bs_results)
axes[0, 1].scatter([batch_sizes[best_idx]], [bs_results[best_idx]], s=200, c='red', 
                  marker='*', zorder=5, label='Best')
axes[0, 1].legend()

# 3. 权重衰减
axes[1, 0].plot(weight_decays, wd_results, '^-', linewidth=2, markersize=10, color='green')
axes[1, 0].set_xlabel('Weight Decay', fontsize=11)
axes[1, 0].set_ylabel('Validation Accuracy', fontsize=11)
axes[1, 0].set_title('权重衰减敏感性', fontsize=13, weight='bold')
axes[1, 0].grid(True, alpha=0.3)

best_idx = np.argmax(wd_results)
axes[1, 0].scatter([weight_decays[best_idx]], [wd_results[best_idx]], s=200, c='red', 
                  marker='*', zorder=5, label='Best')
axes[1, 0].legend()

# 4. 训练轮数
axes[1, 1].plot(epoch_nums, epoch_results, 'd-', linewidth=2, markersize=10, color='purple')
axes[1, 1].set_xlabel('Epochs', fontsize=11)
axes[1, 1].set_ylabel('Validation Accuracy', fontsize=11)
axes[1, 1].set_title('训练轮数敏感性', fontsize=13, weight='bold')
axes[1, 1].grid(True, alpha=0.3)

best_idx = np.argmax(epoch_results)
axes[1, 1].scatter([epoch_nums[best_idx]], [epoch_results[best_idx]], s=200, c='red', 
                  marker='*', zorder=5, label='Best')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# 计算敏感性（标准差）
sensitivities = {
    'Learning Rate': np.std(lr_results),
    'Batch Size': np.std(bs_results),
    'Weight Decay': np.std(wd_results),
    'Epochs': np.std(epoch_results),
}

print("\n超参数敏感性排名（标准差越大越敏感）：")
print("-" * 50)
for param, sensitivity in sorted(sensitivities.items(), key=lambda x: x[1], reverse=True):
    print(f"  {param:<20} {sensitivity:.4f}")

print()
print("核心观察：")
print("- 学习率是最敏感的超参数，需要仔细调整")
print("- 批次大小影响训练稳定性和泛化能力")
print("- 权重衰减帮助防止过拟合")
print("- 训练轮数需要配合早停使用")
print()
print("✓ 超参数敏感性分析完成！")

## 7. 完整微调流程

### 7.1 端到端微调步骤

```
步骤1: 数据准备
├── 数据加载
├── 数据清洗
├── 数据划分（train/val/test）
└── 数据增强（可选）

步骤2: 模型准备
├── 加载预训练模型
├── 添加任务特定层
├── 配置冻结/解冻策略
└── 初始化优化器

步骤3: 训练配置
├── 设置超参数
├── 配置学习率调度
├── 设置早停策略
└── 配置检查点保存

步骤4: 训练循环
├── 前向传播
├── 计算损失
├── 反向传播
├── 参数更新
└── 验证评估

步骤5: 评估与分析
├── 测试集评估
├── 错误分析
├── 可视化结果
└── 模型保存
```

### 7.2 最佳实践

#### 1. 数据处理
- ✅ 使用验证集进行超参数选择
- ✅ 保持测试集完全独立
- ✅ 数据增强提高泛化能力
- ✅ 处理类别不平衡问题

#### 2. 训练技巧
- ✅ 使用梯度裁剪防止梯度爆炸
- ✅ 监控训练和验证损失
- ✅ 使用早停防止过拟合
- ✅ 保存最佳模型检查点

#### 3. 调试策略
- ✅ 先在小数据集上验证流程
- ✅ 检查数据加载是否正确
- ✅ 验证损失是否下降
- ✅ 可视化学习曲线

#### 4. 性能优化
- ✅ 使用混合精度训练（FP16）
- ✅ 梯度累积模拟大批次
- ✅ 数据并行加速训练
- ✅ 优化数据加载流程

In [ ]:
# 🚀 Capstone: 完整的文本分类微调流程
# Goal: 实现端到端的微调pipeline
# Expected outcome: 掌握完整的微调流程

class TextClassificationModel(nn.Module):
    """文本分类模型（模拟BERT结构）"""
    
    def __init__(self, vocab_size=1000, embed_dim=128, hidden_dim=256, num_classes=3, dropout=0.1):
        super(TextClassificationModel, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Encoder (模拟预训练的transformer)
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, input_ids):
        # Embedding
        embedded = self.embedding(input_ids)  # (batch, seq_len, embed_dim)
        
        # Mean pooling
        pooled = embedded.mean(dim=1)  # (batch, embed_dim)
        
        # Encoding
        encoded = self.encoder(pooled)  # (batch, hidden_dim)
        
        # Classification
        logits = self.classifier(encoded)  # (batch, num_classes)
        
        return logits

class FineTuningTrainer:
    """完整的微调训练器"""
    
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
        
        # Loss function
        self.criterion = nn.CrossEntropyLoss()
        
        # Learning rate scheduler
        total_steps = len(train_loader) * config['epochs']
        warmup_steps = int(config['warmup_ratio'] * total_steps)
        self.scheduler = optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=lambda step: min(1.0, step / warmup_steps) if step < warmup_steps 
                                   else max(0.0, (total_steps - step) / (total_steps - warmup_steps))
        )
        
        # Training history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_acc': [],
            'learning_rates': []
        }
        
        # Early stopping
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.best_model_state = None
    
    def train_epoch(self):
        """训练一个epoch"""
        self.model.train()
        total_loss = 0
        
        for batch_idx, (input_ids, labels) in enumerate(self.train_loader):
            # Forward pass
            self.optimizer.zero_grad()
            logits = self.model(input_ids)
            loss = self.criterion(logits, labels)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # Optimizer step
            self.optimizer.step()
            self.scheduler.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(self.train_loader)
        return avg_loss
    
    def validate(self):
        """验证模型"""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for input_ids, labels in self.val_loader:
                logits = self.model(input_ids)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                
                _, predicted = torch.max(logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_loss = total_loss / len(self.val_loader)
        accuracy = correct / total
        
        return avg_loss, accuracy
    
    def train(self):
        """完整训练流程"""
        print("开始训练...")
        print(f"配置: LR={self.config['learning_rate']}, Epochs={self.config['epochs']}, Batch Size={self.config['batch_size']}")
        print("-" * 70)
        
        for epoch in range(self.config['epochs']):
            # Train
            train_loss = self.train_epoch()
            
            # Validate
            val_loss, val_acc = self.validate()
            
            # Record history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['learning_rates'].append(self.optimizer.param_groups[0]['lr'])
            
            # Print progress
            print(f"Epoch {epoch+1}/{self.config['epochs']} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"Val Acc: {val_acc:.4f} | "
                  f"LR: {self.optimizer.param_groups[0]['lr']:.2e}")
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_model_state = self.model.state_dict().copy()
                self.patience_counter = 0
                print("  → 新的最佳模型！")
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config['patience']:
                    print(f"\n早停触发！在epoch {epoch+1}停止训练。")
                    break
        
        # Load best model
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            print("\n已加载最佳模型。")
        
        print("训练完成！")
        return self.history

print("=== 完整微调流程演示 ===")
print()

# 步骤1: 数据准备
print("步骤1: 数据准备")
print("-" * 70)

# 模拟文本数据（token ids）
np.random.seed(42)
torch.manual_seed(42)

# 训练集
train_input_ids = torch.randint(0, 1000, (800, 20))  # 800 samples, seq_len=20
train_labels = torch.randint(0, 3, (800,))  # 3 classes
train_dataset = TensorDataset(train_input_ids, train_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 验证集
val_input_ids = torch.randint(0, 1000, (200, 20))
val_labels = torch.randint(0, 3, (200,))
val_dataset = TensorDataset(val_input_ids, val_labels)
val_loader = DataLoader(val_dataset, batch_size=32)

# 测试集
test_input_ids = torch.randint(0, 1000, (200, 20))
test_labels = torch.randint(0, 3, (200,))
test_dataset = TensorDataset(test_input_ids, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")
print()

# 步骤2: 模型准备
print("步骤2: 模型准备")
print("-" * 70)

model = TextClassificationModel(
    vocab_size=1000,
    embed_dim=128,
    hidden_dim=256,
    num_classes=3,
    dropout=0.1
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print()

# 步骤3: 训练配置
print("步骤3: 训练配置")
print("-" * 70)

config = {
    'learning_rate': 3e-4,
    'batch_size': 32,
    'epochs': 15,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'patience': 3,  # Early stopping patience
}

for key, value in config.items():
    print(f"  {key}: {value}")
print()

# 步骤4: 训练
print("步骤4: 训练")
print("-" * 70)

trainer = FineTuningTrainer(model, train_loader, val_loader, config)
history = trainer.train()
print()

In [ ]:
# 步骤5: 评估与可视化
print("步骤5: 评估与可视化")
print("-" * 70)

# 测试集评估
model.eval()
test_correct = 0
test_total = 0
all_predictions = []
all_labels = []

with torch.no_grad():
    for input_ids, labels in test_loader:
        logits = model(input_ids)
        _, predicted = torch.max(logits, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = test_correct / test_total
print(f"测试集准确率: {test_accuracy:.4f}")
print()

# 可视化训练过程
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# 1. 损失曲线
axes[0, 0].plot(epochs_range, history['train_loss'], 'o-', linewidth=2, 
               markersize=6, label='Train Loss', color='steelblue')
axes[0, 0].plot(epochs_range, history['val_loss'], 's-', linewidth=2, 
               markersize=6, label='Val Loss', color='coral')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('训练和验证损失', fontsize=14, weight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# 标注最佳epoch
best_epoch = np.argmin(history['val_loss']) + 1
axes[0, 0].axvline(x=best_epoch, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0, 0].text(best_epoch, max(history['val_loss']), f'Best\nEpoch {best_epoch}', 
               ha='center', va='top', fontsize=9, color='red', weight='bold')

# 2. 准确率曲线
axes[0, 1].plot(epochs_range, history['val_acc'], '^-', linewidth=2, 
               markersize=8, color='green')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Validation Accuracy', fontsize=12)
axes[0, 1].set_title('验证准确率', fontsize=14, weight='bold')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=test_accuracy, color='red', linestyle='--', 
                  linewidth=2, label=f'Test Acc: {test_accuracy:.3f}')
axes[0, 1].legend(fontsize=11)

# 3. 学习率变化
steps = range(len(history['learning_rates']))
axes[1, 0].plot(steps, history['learning_rates'], linewidth=2, color='purple')
axes[1, 0].set_xlabel('Training Step', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('学习率调度', fontsize=14, weight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_yscale('log')

# 4. 混淆矩阵
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1], 
           cbar_kws={'label': 'Count'})
axes[1, 1].set_xlabel('Predicted Label', fontsize=12)
axes[1, 1].set_ylabel('True Label', fontsize=12)
axes[1, 1].set_title('混淆矩阵', fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

# 性能总结
print("\n性能总结：")
print("=" * 70)
print(f"最佳验证损失: {min(history['val_loss']):.4f} (Epoch {best_epoch})")
print(f"最佳验证准确率: {max(history['val_acc']):.4f}")
print(f"最终测试准确率: {test_accuracy:.4f}")
print(f"训练轮数: {len(history['train_loss'])}")
print()

# 每类性能
from sklearn.metrics import classification_report
print("分类报告：")
print(classification_report(all_labels, all_predictions, 
                          target_names=['Class 0', 'Class 1', 'Class 2']))

print("\n核心观察：")
print("- 训练损失和验证损失都在下降，模型正常收敛")
print("- 早停机制防止了过拟合")
print("- 学习率调度帮助模型稳定训练")
print("- 混淆矩阵显示模型在各类别上的表现")
print()
print("✓ 完整微调流程演示完成！")

## 8. 常见问题与调试

### 8.1 常见问题 FAQ

#### Q1: 如何选择预训练模型？

**A:** 考虑以下因素：

1. **任务类型**
   - 分类/NER → BERT, RoBERTa
   - 生成 → GPT, T5
   - 编码-解码 → BART, T5

2. **语言**
   - 英文 → BERT, RoBERTa, GPT
   - 中文 → BERT-Chinese, RoBERTa-Chinese
   - 多语言 → mBERT, XLM-R

3. **模型大小**
   - 资源充足 → Large模型
   - 资源受限 → Base或Small模型
   - 移动端 → DistilBERT, TinyBERT

4. **领域**
   - 通用 → BERT, RoBERTa
   - 科学 → SciBERT
   - 医学 → BioBERT, PubMedBERT
   - 法律 → LegalBERT

---

#### Q2: 微调时如何避免过拟合？

**A:** 多种策略组合使用：

1. **数据层面**
   - 数据增强（回译、同义词替换）
   - 增加训练数据
   - 使用更多的验证数据

2. **模型层面**
   - 增大Dropout（0.3-0.5）
   - 使用权重衰减（0.01-0.1）
   - 减小模型容量

3. **训练层面**
   - 早停（patience=3-5）
   - 减少训练轮数
   - 使用较小的学习率
   - 特征提取而非全量微调

4. **正则化**
   - L2正则化
   - Label smoothing
   - Mixup/Cutout

---

#### Q3: 如何处理类别不平衡？

**A:** 多种方法：

1. **数据层面**
   - 过采样少数类
   - 欠采样多数类
   - SMOTE等合成方法

2. **损失函数**
   - 类别权重：`nn.CrossEntropyLoss(weight=class_weights)`
   - Focal Loss
   - Class-balanced Loss

3. **评估指标**
   - 使用F1-score而非准确率
   - 关注少数类的召回率
   - 使用宏平均（macro-average）

---

#### Q4: 训练不收敛怎么办？

**A:** 逐步排查：

1. **检查数据**
   - 数据是否正确加载
   - 标签是否正确
   - 是否有数据泄露

2. **调整学习率**
   - 学习率太大 → 减小10倍
   - 学习率太小 → 增大10倍
   - 使用学习率查找器

3. **检查梯度**
   - 梯度爆炸 → 使用梯度裁剪
   - 梯度消失 → 检查激活函数和初始化

4. **简化问题**
   - 先在小数据集上过拟合
   - 验证模型能学习
   - 逐步增加复杂度

---

#### Q5: 什么时候用特征提取，什么时候用微调？

**A:** 决策树：

```
数据量 < 1000？
├── 是 → 特征提取
└── 否 → 继续判断
    |
    计算资源充足？
    ├── 是 → 微调
    └── 否 → 特征提取或部分微调
        |
        任务与预训练相似？
        ├── 是 → 特征提取
        └── 否 → 微调（至少高层）
```

**经验法则**：
- 数据 < 1K → 特征提取
- 数据 1K-10K → 部分微调
- 数据 > 10K → 全量微调

---

#### Q6: 如何加速微调训练？

**A:** 多种优化策略：

1. **混合精度训练**
   ```python
   from torch.cuda.amp import autocast, GradScaler
   scaler = GradScaler()
   
   with autocast():
       outputs = model(inputs)
       loss = criterion(outputs, labels)
   
   scaler.scale(loss).backward()
   scaler.step(optimizer)
   scaler.update()
   ```

2. **梯度累积**
   ```python
   accumulation_steps = 4
   for i, (inputs, labels) in enumerate(dataloader):
       outputs = model(inputs)
       loss = criterion(outputs, labels) / accumulation_steps
       loss.backward()
       
       if (i + 1) % accumulation_steps == 0:
           optimizer.step()
           optimizer.zero_grad()
   ```

3. **数据并行**
   ```python
   model = nn.DataParallel(model)
   ```

4. **优化数据加载**
   ```python
   dataloader = DataLoader(
       dataset,
       batch_size=32,
       num_workers=4,  # 多进程加载
       pin_memory=True  # 加速GPU传输
   )
   ```

---

#### Q7: 如何评估微调效果？

**A:** 多维度评估：

1. **定量指标**
   - 准确率、精确率、召回率、F1
   - 混淆矩阵
   - ROC曲线和AUC

2. **学习曲线**
   - 训练/验证损失曲线
   - 是否过拟合/欠拟合
   - 收敛速度

3. **错误分析**
   - 分析错误样本
   - 找出模型弱点
   - 针对性改进

4. **对比基线**
   - 与从零训练对比
   - 与其他预训练模型对比
   - 与SOTA方法对比

---

#### Q8: 微调后性能反而下降？

**A:** 可能的原因和解决方案：

1. **学习率太大**
   - 破坏了预训练权重
   - 解决：减小学习率（1e-5到5e-5）

2. **过拟合**
   - 在小数据集上训练太久
   - 解决：早停、增大正则化

3. **灾难性遗忘**
   - 忘记了预训练知识
   - 解决：使用更小的学习率、渐进式解冻

4. **数据分布不匹配**
   - 训练集和测试集分布差异大
   - 解决：检查数据划分、使用领域适应

---

### 8.2 调试检查清单

**训练前检查**：
- [ ] 数据加载正确
- [ ] 标签范围正确（0到num_classes-1）
- [ ] 模型输出维度正确
- [ ] 损失函数选择正确
- [ ] 优化器配置正确

**训练中监控**：
- [ ] 训练损失下降
- [ ] 验证损失下降
- [ ] 梯度范数正常（不爆炸/消失）
- [ ] 学习率调度正常
- [ ] GPU利用率高

**训练后分析**：
- [ ] 测试集性能合理
- [ ] 没有严重过拟合
- [ ] 各类别性能均衡
- [ ] 错误样本可解释
- [ ] 与基线对比有提升

## 9. 本章总结

### 9.1 核心要点回顾

#### 1. 迁移学习理论
- ✅ 预训练模型学习了通用的语言表示
- ✅ 迁移学习比从零训练更高效
- ✅ 底层学习通用特征，高层学习任务特定特征

#### 2. 预训练-微调范式
- ✅ 两阶段学习：预训练 + 微调
- ✅ 预训练在大规模无标注数据上
- ✅ 微调在少量任务特定标注数据上

#### 3. 特征提取 vs 微调
- ✅ 特征提取：冻结预训练模型，只训练任务头
- ✅ 微调：更新部分或全部预训练参数
- ✅ 数据少用特征提取，数据多用微调

#### 4. 学习率调度
- ✅ Warmup避免训练初期不稳定
- ✅ 余弦退火实现平滑衰减
- ✅ 判别式学习率：底层小LR，高层大LR
- ✅ 渐进式解冻逐层适应

#### 5. 超参数选择
- ✅ 学习率是最重要的超参数
- ✅ 微调通常用1e-5到5e-5的学习率
- ✅ 批次大小、权重衰减、Dropout需要调整
- ✅ 使用早停防止过拟合

#### 6. 完整流程
- ✅ 数据准备 → 模型准备 → 训练配置 → 训练 → 评估
- ✅ 监控训练和验证曲线
- ✅ 使用混淆矩阵和分类报告
- ✅ 错误分析指导改进

---

### 9.2 实践建议

#### 初学者
1. 从小数据集开始实践
2. 使用推荐的默认超参数
3. 先掌握特征提取
4. 逐步尝试微调

#### 进阶者
1. 实验不同的学习率调度策略
2. 尝试判别式学习率和渐进解冻
3. 进行系统的超参数搜索
4. 深入分析模型行为

#### 专家
1. 设计任务特定的微调策略
2. 结合领域知识优化流程
3. 探索新的正则化方法
4. 贡献开源实现

---

### 9.3 关键公式

**迁移学习**：
$$\theta_{\text{finetuned}} = \arg\min_{\theta} \mathcal{L}_{\text{task}}(\mathcal{D}_{\text{target}}, \theta)$$
其中 $\theta$ 初始化为 $\theta_{\text{pretrained}}$

**Warmup + Cosine调度**：
$$\text{lr}(t) = \begin{cases}
\text{lr}_{\text{base}} \times \frac{t}{T_{\text{warmup}}} & t < T_{\text{warmup}} \\
\text{lr}_{\text{min}} + \frac{1}{2}(\text{lr}_{\text{base}} - \text{lr}_{\text{min}})\left(1 + \cos\left(\frac{t-T_{\text{warmup}}}{T_{\text{total}}-T_{\text{warmup}}}\pi\right)\right) & t \geq T_{\text{warmup}}
\end{cases}$$

**判别式学习率**：
$$\text{lr}_{\text{layer}_i} = \text{lr}_{\text{base}} \times \xi^{L-i}$$

---

### 9.4 下一步学习

完成本章后，建议继续学习：

**Module 5.2: 参数高效微调 (PEFT)**
- LoRA、Adapter等高效方法
- 减少可训练参数到0.1%-1%
- 多任务部署策略

**Module 5.3: 领域适应**
- DAPT和TAPT
- 持续学习
- 灾难性遗忘的解决方案

---

### 9.5 思考题

1. **理论理解**：为什么预训练模型的底层参数应该使用更小的学习率？

2. **实践应用**：如果你有1000个标注样本，应该选择特征提取还是微调？为什么？

3. **对比分析**：Warmup和余弦退火分别解决什么问题？能否只用其中一个？

4. **问题诊断**：如果训练损失下降但验证损失上升，应该如何调整？

5. **方法设计**：设计一个针对极小数据集（<100样本）的微调策略。

---

### 9.6 参考资源

**论文**：
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- [Universal Language Model Fine-tuning (ULMFiT)](https://arxiv.org/abs/1801.06146)
- [How to Fine-Tune BERT for Text Classification](https://arxiv.org/abs/1905.05583)

**博客**：
- [Hugging Face Fine-tuning Guide](https://huggingface.co/docs/transformers/training)
- [Sebastian Ruder's Transfer Learning Guide](https://ruder.io/transfer-learning/)

**代码**：
- [Hugging Face Transformers](https://github.com/huggingface/transformers)
- [PyTorch Examples](https://github.com/pytorch/examples)

---

## 🎉 恭喜完成本章学习！

你已经掌握了：
- ✅ 迁移学习的理论基础
- ✅ 预训练-微调范式
- ✅ 特征提取和微调的区别
- ✅ 学习率调度策略
- ✅ 超参数选择方法
- ✅ 完整的微调流程
- ✅ 常见问题的解决方案

**继续前进，探索参数高效微调（PEFT）的世界！** 🚀

## 思考题参考答案

### 问题 1：为什么预训练模型的底层参数应该使用更小的学习率？

**核心原因：层次化特征的通用性差异**

预训练模型的不同层学到了不同抽象程度的特征：

| 层次 | 学到的内容 | 通用性 | 建议学习率 |
|------|-----------|--------|-----------|
| 底层（1-4层） | 词法、句法、基础语义 | 高（跨任务通用） | 最小（如 1e-5） |
| 中层（5-8层） | 短语、语义关系 | 中等 | 中等（如 3e-5） |
| 顶层（9-12层）| 任务相关的高层语义 | 低（需要适配） | 最大（如 5e-5） |

**数学直觉：**

判别式学习率（ULMFiT提出）的核心公式：
$$\text{lr}_{\text{layer}_i} = \text{lr}_{\text{base}} \times \xi^{L-i}$$

其中 $\xi < 1$（通常取 $0.9$~$0.95$），$L$ 为总层数，$i$ 为当前层序号。

**深层原因：**

1. **避免破坏通用知识**：底层已学到的语言规律（词序、句法）是人类语言的普遍特性，在新任务中无需剧烈调整。较大的学习率会造成"灾难性遗忘"，破坏这些珍贵的预训练表示。

2. **梯度信号的衰减**：反向传播时梯度从顶层流向底层会逐渐减小（梯度消失），底层的梯度信号本身就较弱，过大的学习率会导致底层参数更新噪声过大。

3. **收敛稳定性**：底层参数已经处于较好的初始化状态，微小的更新足以适配新任务，而不会破坏已有的知识结构。

**实验验证**：Howard & Ruder (2018) 在 ULMFiT 论文中证明，判别式学习率相比统一学习率可以将分类准确率提升 1-3 个百分点。

---

### 问题 2：如果你有 1000 个标注样本，应该选择特征提取还是微调？为什么？

**结论：1000 样本通常应选择微调（Fine-tuning），但需要搭配正则化策略。**

**决策框架：**

| 方法 | 适用数据量 | 优点 | 缺点 |
|------|-----------|------|------|
| 特征提取 | <100 样本 | 不过拟合，快速 | 性能上限低 |
| 轻量微调（冻结大部分层） | 100-500 样本 | 平衡性能与过拟合 | 需要调参 |
| 全量微调 | >1000 样本 | 最高性能潜力 | 过拟合风险 |

**1000 样本的具体建议：**

1. **优先选择微调**，1000 样本通常足以提供足够的监督信号来更新所有参数，且预训练权重的强大初始化使得收敛更快、泛化更好。

2. **必须搭配正则化**：
   - Dropout（0.1-0.3）
   - Weight Decay（0.01-0.1）
   - 早停（Early Stopping）
   - 数据增强（同义词替换、回译等）

3. **使用较小学习率**：1e-5 到 3e-5 之间，配合 Warmup（5-10% 步骤数）。

4. **例外情况**（此时考虑特征提取）：
   - 目标域与预训练域差距极大
   - 计算资源极度受限
   - 任务标签噪声较大

**实验对比（SST-2 情感分类，参考 BERT 论文）：**

```
特征提取（1000样本）≈ 83% F1
微调（1000样本）      ≈ 90% F1  (+7%)
全量数据微调          ≈ 93% F1
```

---

### 问题 3：Warmup 和余弦退火分别解决什么问题？能否只用其中一个？

**Warmup 解决的问题：训练初期的不稳定性**

训练开始时，模型参数（特别是新添加的分类头）处于随机初始化状态，梯度方差极大。如果直接使用高学习率，会导致：
- 预训练权重被剧烈破坏
- 损失震荡甚至发散

**Warmup 策略**：
$$\text{lr}(t) = \text{lr}_{\text{max}} \times \frac{t}{T_{\text{warmup}}}, \quad t < T_{\text{warmup}}$$

典型 Warmup 步数 = 总步数的 5-10%。

**余弦退火解决的问题：训练后期的精细收敛**

随着训练进行，模型已接近局部最优，较大的学习率会导致参数在最优点附近震荡，无法进一步降低损失。余弦退火使学习率平滑下降，帮助模型"沉入"更好的局部最优。

$$\text{lr}(t) = \text{lr}_{\text{min}} + \frac{1}{2}(\text{lr}_{\text{max}} - \text{lr}_{\text{min}})\left(1 + \cos\left(\frac{t - T_{\text{warmup}}}{T_{\text{total}} - T_{\text{warmup}}} \pi\right)\right)$$

**能否只用其中一个？**

| 组合 | 效果评估 |
|------|---------|
| 只用 Warmup（之后固定 lr） | 后期无法精细收敛，性能次优 |
| 只用余弦退火（无 Warmup） | 初期学习率较低（余弦从高往低），但仍可能初期不稳定 |
| Warmup + 余弦退火 | 最优组合，覆盖训练全程 |
| 只用固定 lr | 对微调通常不推荐 |

**结论**：两者解决不同阶段的问题，组合使用最佳。如果资源限制只能选一个，在微调场景下 **Warmup 更重要**（保护预训练权重）；长时间训练则**余弦退火更重要**（帮助收敛）。

---

### 问题 4：如果训练损失下降但验证损失上升，应该如何调整？

**这是典型的过拟合（Overfitting）表现，诊断和调整步骤如下：**

**第一步：诊断原因**

```
训练损失 ↓，验证损失 ↑
    ├── 模型容量过大？
    ├── 训练数据太少？
    ├── 学习率过大？
    └── 正则化不足？
```

**第二步：系统性调整方案**

**方案 A：增强正则化**
- 增大 Dropout 比例（0.1 → 0.3）
- 增大 Weight Decay（1e-3 → 1e-2）
- 使用 Label Smoothing（0.1）

**方案 B：调整学习率**
- 降低最大学习率（减半）
- 延长 Warmup 步骤数
- 考虑使用学习率查找（LR Finder）

**方案 C：数据层面**
- 检查数据集标注质量
- 增加数据增强（同义词替换、随机删除词语等）
- 使用交叉验证

**方案 D：早停（Early Stopping）**
- 监控验证损失，若连续 N 个 epoch 不改善则停止
- 保存验证损失最低时的模型权重（checkpoint）

**方案 E：减少可训练参数**
- 冻结底层参数（改为特征提取部分层）
- 减小 batch size（引入更多梯度噪声，有正则化效果）

**优先级建议**：
1. 先启用早停（立即生效）
2. 降低学习率（1/2 倍尝试）
3. 增加 Dropout 和 Weight Decay
4. 最后考虑数据增强

---

### 问题 5：设计一个针对极小数据集（<100 样本）的微调策略。

**背景分析**：100 个样本下全量微调必然严重过拟合，需要极其保守的策略。

**推荐策略：渐进式特征提取 + 轻量微调**

**阶段一：纯特征提取（Epoch 1-3）**
```
冻结所有预训练层参数
只训练分类头（随机初始化）
学习率：1e-3（分类头可用较大 lr）
目标：让分类头收敛到合理状态
```

**阶段二：顶层解冻（Epoch 4-6）**
```
解冻最后 1-2 个 Transformer 层
分类头 lr：1e-4
顶层 lr：1e-5
目标：让顶层特征适配新任务
```

**阶段三：评估是否需要继续解冻**
```
若验证指标仍在提升，可解冻更多层
否则停止，以避免过拟合
```

**其他关键技术选择**：

| 技术 | 配置 | 原因 |
|------|------|------|
| Batch Size | 8-16 | 引入梯度噪声，起正则化作用 |
| Dropout | 0.3-0.5 | 比标准配置更强的正则化 |
| Weight Decay | 0.1 | 防止参数爆炸 |
| 数据增强 | 同义词替换、回译 | 人工扩充数据集 |
| 交叉验证 | 5-fold | 避免验证集偏差 |
| 模型集成 | 3-5 个 checkpoint | 减少预测方差 |

**高级选项：Few-shot 学习**

当样本极少时，可考虑：
- **Prompt-based Fine-tuning**：将分类任务转换为语言模型任务（如 PET/iPET）
- **ProtoNet**：计算类原型向量进行分类
- **Meta-learning**（MAML）：学习如何快速学习

**预期性能对比**（100 样本，SST-2）：
```
全量微调（无正则化）   ≈ 72%  （过拟合）
特征提取              ≈ 81%
渐进式解冻策略        ≈ 85%
Prompt-based 方法     ≈ 87%
```

**核心原则**：数据越少，越应该依赖预训练知识，越需要限制参数更新范围。